<a href="https://colab.research.google.com/github/prishaa09/solarflarephase2/blob/main/Solar_Flare_Phase_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ------- setup (run once per session in Colab) -------
!pip uninstall -y numpy pandas
!pip install -q "sunpy==5.1" "numpy<2" pandas astropy

from pathlib import Path
import numpy as np
import pandas as pd
from sunpy.net import Fido, attrs as a
from sunpy import timeseries as ts

# --------------- CONFIG ----------------
START = "2015-01-01 00:00"     # <-- change as needed
END   = "2015-03-31 23:59"     # <-- change as needed
OUT_DIR = Path("/content/data_goes")   # explicit local path
CSV_NAME = f"goes_xrs_{START[:10]}_to_{END[:10]}_1min.csv"
CADENCE = "1min"               # standard for modeling
MIN_ALLOWED = 0.0              # drop non-physical/negatives
# --------------------------------------

# Data fetching and cleaning functions
# Fetch function
def fetch_goes_xrs(start: str, end: str):
    """
    Download GOES XRS science-quality data for the time range.
    Returns a single SunPy TimeSeries (concatenated if multiple files).
    """
    res = Fido.search(a.Time(start, end), a.Instrument("XRS"))
    files = Fido.fetch(res)
    goes_ts = ts.TimeSeries(files)
    # Sometimes a list of TimeSeries is returned (one per file) — concatenate:
    if isinstance(goes_ts, list):
        goes_ts = ts.concatenate(goes_ts)
    return goes_ts


def clean_to_dataframe(goes_ts: ts.TimeSeries, cadence="1min") -> pd.DataFrame:
    """
    Convert to tidy DataFrame, resample, apply basic quality filter,
    and compute simple derived features.
    """
    df = goes_ts.to_dataframe()

    # Standardize column names across GOES generations
    rename_map = {}
    for col in df.columns:
        cl = col.lower()
        if ("1-8" in cl) or ("xrsb" in cl) or ("long" in cl):
            rename_map[col] = "flux_1_8A"
        if ("0.5-4" in cl) or ("xrsa" in cl) or ("short" in cl):
            rename_map[col] = "flux_0.5_4A"
        if ("quality" in cl) or ("qc_flag" in cl):
            rename_map[col] = "quality_flag"
        if "sat" in cl:
            rename_map[col] = "satellite_id"
    df.rename(columns=rename_map, inplace=True)

    # Drop negatives; coerce to numeric
    for c in ("flux_0.5_4A", "flux_1_8A"):
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df.loc[df[c] < MIN_ALLOWED, c] = np.nan

    # Keep only good/unknown quality
    if "quality_flag" in df:
        df = df[df["quality_flag"].isna() | (df["quality_flag"] == 0)]

    # Uniform cadence
    df = df.resample(cadence).mean()

    # Derived features
    if {"flux_0.5_4A", "flux_1_8A"}.issubset(df.columns):
        df["flux_ratio_A_over_B"] = df["flux_0.5_4A"] / df["flux_1_8A"]
        for win in [5, 15, 30]:  # minutes
            df[f"long_mean_{win}m"]  = df["flux_1_8A"].rolling(f"{win}min").mean()
            df[f"long_std_{win}m"]   = df["flux_1_8A"].rolling(f"{win}min").std()
            df[f"short_mean_{win}m"] = df["flux_0.5_4A"].rolling(f"{win}min").mean()

    # Require at least one flux present
    df = df.dropna(subset=[c for c in ["flux_1_8A", "flux_0.5_4A"] if c in df.columns], how="all")

    # Flatten index
    df = df.reset_index().rename(columns={"index": "time_utc"})
    return df


def save_csv(df: pd.DataFrame, out_dir: Path, name: str) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / name
    df.to_csv(out_path, index=False)
    print(f"✅ Saved: {out_path.resolve()}")
    return out_path


# ------------------ RUN -------------------
print(f"Fetching GOES XRS from {START} to {END} …")
goes_ts = fetch_goes_xrs(START, END)
df = clean_to_dataframe(goes_ts, cadence=CADENCE)
csv_path = save_csv(df, OUT_DIR, CSV_NAME)

# Show a peek + where it is
print(df.head(5))
print(df.describe(include='all'))
!ls -lh /content/data_goes

# ----- Directly download the CSV to your computer -----
from google.colab import files
files.download(str(csv_path))

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.3.3
Uninstalling pandas-2.3.3:
  Successfully uninstalled pandas-2.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but y

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject